In [1]:
import numpy as np

### sample amount for min detectable effect

In [ ]:
import math

In [1]:
import noshmishmosh

In [16]:
def get_sample_size(minimum_detectable, significance, conversion_rate):
    # Default z-score for 0.05 significance (95% confidence)
    zscore = 1.959964
    
    if significance == 0.15:
        zscore = 1.439531
    elif significance == 0.1:
        zscore = 1.644854

    factor = 2 * (0.8416212 + zscore)**2
    conv_b = conversion_rate * (1 + minimum_detectable)

    numerator = math.sqrt(conversion_rate * (1 - conversion_rate) + conv_b * (1 - conv_b))
    denominator = abs(conversion_rate * minimum_detectable)
    
    min_sample_size = factor * (numerator / denominator)**2

    # Formatting to 2 significant figures and converting back to float
    return float(f"{min_sample_size:.2g}")

In [22]:
all_visitors = noshmishmosh.customer_visits
paying_visitors = noshmishmosh.purchasing_customers

total_visitor_count = len(all_visitors)
total_visitor_count

500

In [4]:
paying_visitor_count = len(paying_visitors)
paying_visitor_count

93

In [5]:
baseline_percent = paying_visitor_count / total_visitor_count * 100.0
print(baseline_percent)

18.6


In [9]:
payment_history = noshmishmosh.money_spent
average_payment = np.mean(payment_history)
print(average_payment)

26.543655913978498


In [10]:
new_customers_needed = np.ceil(1240 / average_payment)
print(new_customers_needed)

47.0


In [11]:
percentage_point_increase = new_customers_needed / total_visitor_count * 100.0
print(percentage_point_increase)

9.4


In [12]:
# Minimum detectable effect:
mde = percentage_point_increase / baseline_percent * 100.0
print(mde)

50.53763440860215


In [18]:
sig = 0.1

In [21]:
get_sample_size(mde*0.01, sig, baseline_percent*0.01)

490.0

-> amount of samples we need to make meaningful decisions

### Game purchase rate

FarmBurg, a company that makes a farming simulation social network game. In the FarmBurg game, you can plow, plant, and harvest different crops. We need to A/B test for pricing of an upgrade package.

In [2]:
import pandas as pd

In [4]:
from scipy.stats import chi2_contingency, binomtest

In [5]:
abdata = pd.read_csv('clicks.csv')
abdata.head()

,user_id,group,is_purchase
0,8e27bf9a,A,No
1,eb89e6f0,A,No
2,7119106a,A,No
3,e53781ff,A,No
4,02d48cf1,A,Yes


- user_id: a unique id for each visitor to the FarmBurg site
- group: either 'A', 'B', or 'C' depending on which group the visitor was assigned to
- is_purchase: either 'Yes' if the visitor made a purchase or 'No' if they did not.

In [6]:
Xtab = pd.crosstab(abdata.group, abdata.is_purchase)
Xtab

is_purchase,No,Yes
group,,
A,1350,316
B,1483,183
C,1583,83


Group A has the highest number of purchases, with 316 purchases.

In [8]:
chi2, pval, dof, expected = chi2_contingency(Xtab)
print(pval)

2.4126213546684264e-35


-> There is a significant difference in the purchase rate for groups A, B, and C.

In [9]:
num_visits = len(abdata)
print(num_visits)

4998


We tested three different price points: 
- $0.99 (group A)
  
- $1.99 (group B)
  
- $4.99 (group C).

we need to generate a minimum of $1000 in revenue per week in order to justify this project.

In [10]:
num_sales_needed_099 = 1000 / 0.99
print(num_sales_needed_099)

num_sales_needed_199 = 1000 / 1.99
print(num_sales_needed_199)

num_sales_needed_499 = 1000 / 4.99
print(num_sales_needed_499)

p_sales_needed_099 = num_sales_needed_099 / num_visits
print(p_sales_needed_099)

p_sales_needed_199 = num_sales_needed_199 / num_visits
print(p_sales_needed_199)

p_sales_needed_499 = num_sales_needed_499 / num_visits
print(p_sales_needed_499)

1010.1010101010102
502.51256281407035
200.40080160320642
0.20210104243717691
0.10054272965467594
0.040096198800161346


In [11]:
samp_size_099 = np.sum(abdata.group == 'A')
sales_099 = np.sum((abdata.group == 'A') & (abdata.is_purchase == 'Yes'))
print(samp_size_099)
print(sales_099)

1666
316


In [13]:
samp_size_199 = 1483 + 183
sales_199 = 183

samp_size_499 = 1583 + 83
sales_499 = 83

In [15]:
binomtest(sales_099, samp_size_099, p_sales_needed_099, alternative = 'greater')

BinomTestResult(k=316, n=1666, alternative='greater', statistic=0.18967587034813926, pvalue=0.9028081076188554)

In [16]:
binomtest(sales_199, samp_size_199, p_sales_needed_199, alternative = 'greater')

BinomTestResult(k=183, n=1666, alternative='greater', statistic=0.10984393757503001, pvalue=0.11184562623740596)

In [17]:
binomtest(sales_499, samp_size_499, p_sales_needed_499, alternative = 'greater')

BinomTestResult(k=83, n=1666, alternative='greater', statistic=0.04981992797118848, pvalue=0.027944826659830616)

C group is the only group where we would conclude that the purchase rate is significantly higher than the target needed to reach $1000 revenue per week. 

Therefore, we should charge $4.99 for the upgrade.